## Imports

In [0]:
import requests
import zipfile
import os
import shutil
from io import BytesIO

## Setting Up Paths

In [0]:
volume_path = "/Volumes/bts_flight_data/bronze/bts_flight_dataset"
zip_dir = f"{volume_path}/zipped"
extract_dir = f"{volume_path}/unzipped"

os.makedirs(zip_dir, exist_ok=True)
os.makedirs(extract_dir, exist_ok=True)

lookups = {
    "airports": "https://raw.githubusercontent.com/jpatokal/openflights/master/data/airports.dat",
    "carriers": "https://raw.githubusercontent.com/jpatokal/openflights/master/data/airlines.dat",
}

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

## Downloading the Lookup Files

In [0]:
for name, url in lookups.items():
    resp = requests.get(url, headers=headers, verify=False)

    if resp.status_code != 200:
        print(f"✗ {name} failed — status {resp.status_code}")
        continue

    # Check if the response content is actually a zip, by magic bytes (more reliable than trusting the URL/extension)
    is_zip = resp.content[:2] == b'PK'

    if is_zip:
        zip_dest = f"{zip_dir}/lookup_{name}.zip"
        with open(zip_dest, "wb") as f:
            f.write(resp.content)
        print(f"✓ {name} is a zip → saved to {zip_dest}")

        with zipfile.ZipFile(zip_dest) as zf:
            zf.extractall(extract_dir)
        print(f"  ↳ Extracted to {extract_dir}")

    else:
        # Not a zip — check it's not HTML before trusting it as CSV/data
        preview = resp.content[:200].decode(errors="ignore")
        if "<!DOCTYPE" in preview or "<html" in preview.lower():
            print(f"✗ {name} returned HTML, not real data — status {resp.status_code}")
            print("Preview:", preview)
            continue

        dest = f"{extract_dir}/lookup_{name}.csv"
        with open(dest, "wb") as f:
            f.write(resp.content)
        print(f"✓ {name} is plain data → saved directly to {dest}")

## Cleaning Columns

In [0]:
import re

def clean_columns(df):
    new_cols = [re.sub(r'[ ,;{}()\n\t=]', '_', c) for c in df.columns]
    return df.toDF(*new_cols)

## Loading into Bronze Delta Table

In [0]:
%sql
DROP TABLE IF EXISTS bts_flight_data.bronze.airports_lookup;
DROP TABLE IF EXISTS bts_flight_data.bronze.carriers_lookup;

In [0]:
volume_path = "/Volumes/bts_flight_data/bronze/bts_flight_dataset"
extract_dir = f"{volume_path}/unzipped"

# --- Airports ---
df_airports = spark.read.csv(
    f"{extract_dir}/lookup_airports.csv",
    header=False,
    inferSchema=True
)
df_airports = df_airports.toDF(
    "airport_id", "name", "city", "country", "iata", "icao",
    "latitude", "longitude", "altitude", "timezone", "dst",
    "tz_database_timezone", "type", "source"
)
df_airports.write.format("delta").mode("overwrite").saveAsTable("bts_flight_data.bronze.airports_lookup")

# --- Carriers ---
df_carriers = spark.read.csv(
    f"{extract_dir}/lookup_carriers.csv",
    header=False,
    inferSchema=True
)
df_carriers = df_carriers.toDF(
    "airline_id", "name", "alias", "iata", "icao", "callsign", "country", "active"
)
df_carriers.write.format("delta").mode("overwrite").saveAsTable("bts_flight_data.bronze.carriers_lookup")

# --- Verify ---
print("Airports:", df_airports.count())
print("Carriers:", df_carriers.count())

## Sanity Check

In [0]:
%sql
LIST '/Volumes/bts_flight_data/bronze/bts_flight_dataset';

In [0]:
%sql
LIST '/Volumes/bts_flight_data/bronze/bts_flight_dataset/unzipped';

In [0]:
%sql
SELECT 'airports' AS source, COUNT(*) AS row_count FROM bts_flight_data.bronze.airports_lookup
UNION ALL
SELECT 'carriers' AS source, COUNT(*) AS row_count FROM bts_flight_data.bronze.carriers_lookup;

In [0]:
%sql
SELECT COUNT(*) AS total_flights
FROM bts_flight_data.bronze.flights_raw;

In [0]:
%sql
SELECT year, month, COUNT(*) AS flight_count
FROM bts_flight_data.bronze.flights_raw
GROUP BY year, month
ORDER BY year, month;